In [38]:
'''
K-Means++ Clustering
'''
from numpy.random import uniform
import numpy as np
import yfinance as yf
import pandas as pd
from scipy.spatial import distance
import math
import numbers
import seaborn as sns
import matplotlib.pyplot as plt

def get_correlation(data):
  return data.corr(method='pearson')

def calc_distance(correlation):
  distance_corr = np.sqrt(0.5 * (1 - correlation))
  return distance_corr

def euclidian_distance_improve(len_stocks, distance_df):
  eucli_dist = pd.DataFrame()
  for n in range(len_stocks):
    for k in range(( distance_df.shape[0] * distance_df.shape[1] )):
      row = int(np.floor(k / distance_df.shape[1]))
      col = k % distance_df.shape[1]
      eucli_dist.loc[row, col] = np.sqrt((distance_df.iloc[n, row] - distance_df.iloc[n, col]) ** 2)
  eucli_dist.index = distance_df.index
  eucli_dist.columns = distance_df.columns
  return eucli_dist

def euclidean(point, data):
    """
    Euclidean distance between point & data.
    Point has dimensions (m,), data has dimensions (n,m), and output will be of size (n,).
    """
    return np.sqrt(np.sum((point - data)**2, axis=1))

import numpy as np
import random
import numbers
from scipy.spatial.distance import cdist

class KMeans:
    def __init__(self, n_clusters=8, random_state=None, max_iter=300):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.labels_ = None  # Atributo para armazenar as labels

    def fit(self, X_train):
        sample_weight = None
        random_state = self.check_random_state(self.random_state)
        sample_weight = self._check_sample_weight(sample_weight, X_train, dtype=X_train.dtype)
        self.centroids = self.centroids(X_train, random_state, sample_weight)

        iteration = 0
        prev_centroids = None
        while np.not_equal(self.centroids, prev_centroids).any() and iteration < self.max_iter:
            sorted_points = [[] for _ in range(self.n_clusters)]

            for x in X_train:
                dists = self.euclidean(x, self.centroids)
                centroid_idx = np.argmin(dists)
                sorted_points[centroid_idx].append(x)

            prev_centroids = self.centroids.copy()
            self.centroids = [np.mean(cluster, axis=0) for cluster in sorted_points]

            for i, centroid in enumerate(self.centroids):
                if np.isnan(centroid).any():
                    self.centroids[i] = prev_centroids[i]

            iteration += 1

        # Após o fit, armazenamos as labels
        self.labels_ = self.assign_labels(X_train)  # Armazenando as labels
        return self

    def assign_labels(self, X_train):
        """Atribui os rótulos (labels) de cada ponto aos seus clusters"""
        labels = []
        for x in X_train:
            dists = self.euclidean(x, self.centroids)
            centroid_idx = np.argmin(dists)
            labels.append(centroid_idx)
        return np.array(labels)

    def check_random_state(self, seed):
        if seed is None or seed is np.random:
            return np.random.mtrand._rand
        if isinstance(seed, numbers.Integral):
            return np.random.RandomState(seed)
        if isinstance(seed, np.random.RandomState):
            return seed

    def _check_sample_weight(self, sample_weight, X_train, dtype):
        n_samples = X_train.shape[0]
        if sample_weight is None:
            sample_weight = np.ones(n_samples, dtype=X_train.dtype)
        return sample_weight

    def centroids(self, X_train, random_state, sample_weight):
        n_samples, n_features = X_train.shape
        centers = np.empty((self.n_clusters, n_features), dtype=X_train.dtype)
        n_local_trials = 2 + int(np.log(self.n_clusters))
        center_id = random_state.choice(n_samples, p=sample_weight / sample_weight.sum())
        centers[0] = X_train[center_id]

        for c in range(1, self.n_clusters):
            distances = np.array([min(np.linalg.norm(x - c)**2 for c in centers) for x in X_train])
            probabilities = distances / distances.sum()
            cumulative_probabilities = probabilities.cumsum()
            r = random_state.uniform(size=n_local_trials)

            for i, p in enumerate(cumulative_probabilities):
                if (r < p).any():
                    centers[c] = X_train[i]
                    break
        return np.array(centers)

    def euclidean(self, x, centroids):
        return np.linalg.norm(x - centroids, axis=1)

    def get_labels(self):
        """Retorna as labels dos clusters após o fit"""
        return self.labels_

    def evaluate(self, X):
        centroids = []
        centroid_idxs = []
        for x in X:
            dists = self.euclidean(x, self.centroids)
            centroid_idx = np.argmin(dists)
            centroids.append(self.centroids[centroid_idx])
            centroid_idxs.append(centroid_idx)
        return centroids, centroid_idxs

asset = ['MSFT', 'PCAR', 'JPM', 'AAPL', 'GOOGL', 'AMZN', 'ITUB', 'VALE', 'SHEL', 'BAC']
start = '2016-01-01'; end = '2022-01-01'
data = yf.download(asset, start, end)['Adj Close']
correlation = get_correlation(data)
distance_corr = calc_distance(correlation)
distance_euclidean = euclidian_distance_improve(len(asset), distance_corr)

centers = 3
X_train = distance_euclidean.values
# Fit centroids to dataset
kmeans = KMeans(n_clusters=centers, random_state=42)
kmeans.fit(X_train)

labels = kmeans.get_labels()
print(labels)

class_centers, classification = kmeans.evaluate(X_train)
print(classification)
#print("Clusters:")
#for ticker, label in zip(distance_euclidean.index, classification):
#    print(f"{ticker}: Cluster {label}")
#sns.scatterplot(x=[X[0] for X in X_train],
#                y=[X[1] for X in X_train],
#                hue=classification,
#                style=classification,
#                palette="deep",
#                legend=None
#                )
#mean_centroids = np.mean(kmeans.centroids, axis=1)
# Para plotar, vamos pegar as duas primeiras dimensões de cada centróide
# Configurando o gráfico
#plt.plot(mean_centroids, 'k+', markersize=10,)
#plt.show()

[*********************100%***********************]  10 of 10 completed


[0 0 2 0 1 2 0 0 1 2]
[0, 0, 2, 0, 1, 2, 0, 0, 1, 2]


In [39]:
'''
X-Means usando o meu KMeans++
'''
from scipy.stats import multivariate_normal
class XMeans:
  def __init__(self, k_min=1, k_max=10, max_iter=100):
    self.data = data
    self.k_min = k_min
    self.k_max = k_max
    self.max_iter = max_iter

  def fit(self, data):
    current_k = self.k_min
    # Passo 2
    # Realizando KMeans com numero de clusters igual a 2 em todo o dado
    kmeans = KMeans(n_clusters=current_k, random_state=42, max_iter=300)
    kmeans.fit(data)

    best_labels = kmeans.labels_
    best_centroids = kmeans.centroids

    # Passo 5
    # Calcular o bic para o cluster Ci
    best_bic = self.calculate_bic(data, best_labels, best_centroids)
    
    # Passo 3
    # Repetir o processo do passo 4 até o passo 8
    while current_k < self.k_max:
        candidate_k = current_k + 1
        
        # Passo 4
        # Aplicar KMeans ao cluster Ci
        kmeans = KMeans(n_clusters=candidate_k, random_state=42, max_iter=300)
        kmeans.fit(data)

        candidate_labels = kmeans.labels_
        candidate_centroids = kmeans.centroids

        # Passo 6
        # Calcular o bic para o cluster Ci_1 e Ci_2
        candidate_bic = self.calculate_bic(data, candidate_labels, candidate_centroids)

        # Passo 7
        # Se BIC > BIC' então preferimos a divisão e continuamos o codigo
        if candidate_bic > best_bic:
            best_bic = candidate_bic
            best_labels = candidate_labels
            best_centroids = candidate_centroids
            current_k = candidate_k
        
        # Passo 8 
        # Se BIC < BIC' então preferimos não dividir e paramos o codigo
        else:
            break
    return best_labels, best_centroids, best_bic

  def calculate_bic(self, data, labels, centroids):
    n_clusters = len(centroids)
    n_samples, n_features = data.shape
    bic = 0

    for i in range(n_clusters):
        # Obtendo os clusters Ci_1 | Ci_2
        cluster_points = data[labels == i]
        n_points = len(cluster_points)

        # Se numero de elementos dentro do cluster > 1 calculamos o seu bic
        if n_points > 1:
            covariance = np.cov(cluster_points, rowvar=False)
            covariance = np.diag(covariance) if len(covariance.shape) == 1 else covariance
            
            # Verificamos se a matriz covariance é positiva definida
            # Se não for, então forçamos ela a ser no campo except
            try:
                temp = np.linalg.inv(covariance)
            except:
                covariance = self.validate_and_fix_covariance(covariance)
            
            # Calculando o f(x, theta), distribuição normal multivariada
            likelihood = multivariate_normal.logpdf(
                cluster_points, mean=centroids[i], cov=covariance
            ).sum()
            bic += likelihood - 0.5 * n_points * np.log(n_samples)

    return bic

  def validate_and_fix_covariance(self, cov_matrix):
    # Verifica se a matriz é simétrica
    if not np.allclose(cov_matrix, cov_matrix.T):
        cov_matrix = (cov_matrix + cov_matrix.T) / 2
    # Verifica se a matriz é positiva definida
    try:
        np.linalg.cholesky(cov_matrix)
    except np.linalg.LinAlgError:
        # Adiciona um pequeno valor à diagonal
        jitter = 1e-6
        cov_matrix += np.eye(cov_matrix.shape[0]) * jitter

        # Verifica novamente
        try:
            np.linalg.cholesky(cov_matrix)
        except np.linalg.LinAlgError:
            raise ValueError("Não foi possível corrigir a matriz para torná-la positiva definida.")
    return cov_matrix

if __name__ == "__main__":
    xm = XMeans()
    labels, centroids, bic = xm.fit(X_train)

    print(f'Labels: {labels}')

Labels: [0 0 2 0 1 2 0 0 1 2]


In [40]:
'''
X-Means usando KMeans++
'''

import numpy as np
from sklearn.cluster import KMeans
from scipy.stats import multivariate_normal
class XMeans:
  def __init__(self, k_min=1, k_max=10, max_iter=100):
    self.data = data
    self.k_min = k_min
    self.k_max = k_max
    self.max_iter = max_iter

  def fit(self, data):
    current_k = self.k_min
    kmeans = KMeans(n_clusters=current_k, random_state=42, max_iter=300)
    kmeans.fit(data)

    best_labels = kmeans.labels_
    best_centroids = kmeans.cluster_centers_
    best_bic = self.calculate_bic(data, best_labels, best_centroids)

    while current_k < self.k_max:
        candidate_k = current_k + 1
        kmeans = KMeans(n_clusters=candidate_k, random_state=42, max_iter=300)
        kmeans.fit(data)

        candidate_labels = kmeans.labels_
        candidate_centroids = kmeans.cluster_centers_
        candidate_bic = self.calculate_bic(data, candidate_labels, candidate_centroids)

        if candidate_bic > best_bic:
            best_bic = candidate_bic
            best_labels = candidate_labels
            best_centroids = candidate_centroids
            current_k = candidate_k
        else:
            break

    return best_labels, best_centroids, best_bic
    # Função para calcular o BIC
  def calculate_bic(self, data, labels, centroids):
    n_clusters = len(centroids)
    n_samples, n_features = data.shape
    bic = 0

    for i in range(n_clusters):
        cluster_points = data[labels == i]
        n_points = len(cluster_points)

        if n_points > 1:
            covariance = np.cov(cluster_points, rowvar=False)
            covariance = np.diag(covariance) if len(covariance.shape) == 1 else covariance
            try:
                temp = np.linalg.inv(covariance)
            except:
                covariance = self.validate_and_fix_covariance(covariance)

            likelihood = multivariate_normal.logpdf(
                cluster_points, mean=centroids[i], cov=covariance
            ).sum()
            bic += likelihood - 0.5 * n_points * np.log(n_samples)

    return bic

  def validate_and_fix_covariance(self, cov_matrix):
    # Verifica se a matriz é simétrica
    if not np.allclose(cov_matrix, cov_matrix.T):
        cov_matrix = (cov_matrix + cov_matrix.T) / 2
    # Verifica se a matriz é positiva definida
    try:
        np.linalg.cholesky(cov_matrix)
    except np.linalg.LinAlgError:
        # Adiciona um pequeno valor à diagonal
        jitter = 1e-6
        cov_matrix += np.eye(cov_matrix.shape[0]) * jitter

        # Verifica novamente
        try:
            np.linalg.cholesky(cov_matrix)
        except np.linalg.LinAlgError:
            raise ValueError("Não foi possível corrigir a matriz para torná-la positiva definida.")
    return cov_matrix

if __name__ == "__main__":
    xm = XMeans()
    labels, centroids, bic = xm.fit(X_train)

    print(f'Labels: {labels}')

Labels: [0 0 2 0 1 2 0 0 1 2]
